# Dynamic Entité Incident Table
Upload a JSON file of incidents. Use the dropdown to select an entity and view their incident repartition by category.

In [1]:
import pandas as pd
import json
import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

!pip install ipywidgets -q

def load_json_to_df(path):
    for encoding in ('utf-8', 'latin-1'):
        try:
            with open(path, encoding=encoding) as f:
                data = json.load(f)
            return pd.json_normalize(data)
        except Exception:
            continue
    raise ValueError(f'Unable to load JSON file: {path}')

def normalize_columns(df):
    mapping = {}
    for col in df.columns:
        normalized = re.sub(r'[^a-z0-9]', '', str(col).lower())
        if normalized in ('client', 'clientname', 'tenant', 'societe', 'company', 'entite', 'entité'):
            mapping[col] = 'Entité'
        elif normalized in ('serveur', 'server', 'host', 'hostname', 'apparail'):
            mapping[col] = 'Serveur'
        elif normalized in ('titre', 'title', 'subject', 'objet', 'description', 'summary'):
            mapping[col] = 'Titre'
    df = df.rename(columns=mapping)
    for target in ('Entité', 'Serveur', 'Titre'):
        if target not in df.columns:
            df[target] = pd.NA
    return df

def categorize_row(row):
    titre = str(row.get('Titre', '') or '')
    serveur = str(row.get('Serveur', '') or '')
    combined = f'{titre} {serveur}'
    if re.search(r'ntnx|nutanix', combined, flags=re.I):
        return 'Nutanix'
    if re.search(r'cpu', titre, flags=re.I):
        return 'CPU Issue'
    if re.search(r'memory|ram', titre, flags=re.I):
        return 'Memory Issue'
    if re.search(r'disk|storage', titre, flags=re.I):
        return 'Disk'
    if re.search(r'network|nic|link', titre, flags=re.I):
        return 'Network'
    return 'OS / Autres'


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: c:\Users\aelazmi2\Desktop\glpi_pipeline.py\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
# Enter the path to your JSON file and run this cell
import os

json_path = "Synthèse DC - Incident.json"  # Change this to your JSON file path

if os.path.exists(json_path):
    df = load_json_to_df(json_path)
    df = normalize_columns(df)
    df['Categorie'] = df.apply(categorize_row, axis=1)
    entities = sorted(df['Entité'].dropna().unique())
    print(f"✅ Loaded {len(df)} incidents")
    print(f"📋 Found {len(entities)} entités")
    entity_dropdown = widgets.Dropdown(description='Entité:', options=entities)
    entity_dropdown.df = df
    entity_dropdown.value = entities[0] if entities else None
    entity_dropdown.value = entities[0] if entities else None
    entity_dropdown.df = df
else:
    print(f"❌ File not found: {json_path}")
    print("Please update the json_path variable above with your JSON file path")

✅ Loaded 29122 incidents
📋 Found 10 entités


In [3]:
# Dropdown and dynamic table
output = widgets.Output()

def update_table(change):
    output.clear_output()
    df = getattr(entity_dropdown, 'df', None)
    if df is None or not entity_dropdown.value:
        return
    filtered = df[df['Entité'] == entity_dropdown.value]
    category_order = [
        'CPU Issue', 'Memory Issue', 'Disk', 'Network', 'Nutanix', 'OS / Autres'
    ]
    display_map = {
        'CPU Issue': 'CPU Issue',
        'Memory Issue': 'Memory Issue',
        'OS / Autres': 'OS service',
        'Nutanix': 'Nutanix Issue',
        'Disk': 'VM availability / Disk / MSSQL / autres',
        'Network': 'Network',
    }
    counts = filtered['Categorie'].value_counts().reindex(category_order, fill_value=0)
    total = counts.sum()
    # Convert Series to DataFrame explicitly to avoid KeyError
    counts_df = counts.reset_index()
    counts_df.columns = ['Categorie', 'Volume']
    table_df = (
        counts_df.assign(Categorie=lambda df: df['Categorie'].map(display_map).fillna(df['Categorie']))
        .assign(Part_du_total=lambda df: (df['Volume'] / total * 100).round(0).astype(int).astype(str) + ' %')
        .loc[lambda df: df['Categorie'].isin([
            'CPU Issue', 'Memory Issue', 'OS service', 'Nutanix Issue', 'VM availability / Disk / MSSQL / autres'
        ])]
        .loc[:, ['Categorie', 'Volume', 'Part_du_total']]
    )
    with output:
        display(HTML(table_df.to_html(index=False, escape=False)))

entity_dropdown.observe(update_table, names='value')
display(entity_dropdown, output)

# Initial display
update_table(None)

Dropdown(description='Entité:', options=('Data_Center > Avocat', 'Data_Center > CG Park', 'Data_Center > CMS',…

Output()

# Export to PowerPoint
Export the dynamic table to a PowerPoint file with all entity data embedded.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import os

def export_to_ppt(df, entities, output_path="stable_analysis.pptx"):
    """Export all entity data to PowerPoint with embedded tables."""
    
    prs = Presentation()
    prs.slide_width = Inches(13.333)  # 16:9 widescreen
    prs.slide_height = Inches(7.5)
    
    # Title slide
    title_slide_layout = prs.slide_layouts[0]
    slide = prs.slides.add_slide(title_slide_layout)
    title = slide.shapes.title
    title.text = "Stable Analysis - Incident Report"
    subtitle = slide.placeholders[1]
    subtitle.text = f"Total: {len(df)} incidents across {len(entities)} entities"
    
    # Define category order and display map
    category_order = ['CPU Issue', 'Memory Issue', 'Disk', 'Network', 'Nutanix', 'OS / Autres']
    display_map = {
        'CPU Issue': 'CPU Issue',
        'Memory Issue': 'Memory Issue',
        'OS / Autres': 'OS service',
        'Nutanix': 'Nutanix Issue',
        'Disk': 'VM availability / Disk / MSSQL / autres',
        'Network': 'Network',
    }
    
    # Create one slide per entity (fast and simple)
    for entity in entities:
        entity_df = df[df['Entité'] == entity]
        counts = entity_df['Categorie'].value_counts().reindex(category_order, fill_value=0)
        total = counts.sum()
        
        # Add blank slide
        blank_layout = prs.slide_layouts[6]  # blank layout
        slide = prs.slides.add_slide(blank_layout)
        
        # Add title
        left = Inches(0.5)
        top = Inches(0.3)
        width = Inches(12.333)
        height = Inches(0.8)
        title_box = slide.shapes.add_textbox(left, top, width, height)
        tf = title_box.text_frame
        p = tf.paragraphs[0]
        p.text = f"Entity: {entity}"
        p.font.size = Pt(28)
        p.font.bold = True
        p.font.color.rgb = RGBColor(0, 51, 102)
        
        # Create table data
        table_data = [['Category', 'Volume', 'Part']]
        for cat in category_order:
            vol = counts.get(cat, 0)
            if vol > 0:
                display_cat = display_map.get(cat, cat)
                pct = f"{int(vol/total*100)}%" if total > 0 else "0%"
                table_data.append([display_cat, str(vol), pct])
        
        # Add table
        table_rows = len(table_data)
        table_cols = 3
        left = Inches(1)
        top = Inches(1.5)
        width = Inches(11.333)
        height = Inches(0.8 * table_rows)
        
        table = slide.shapes.add_table(table_rows, table_cols, left, top, width, height).table
        
        # Style and fill table
        for i, row in enumerate(table_data):
            for j, cell_text in enumerate(row):
                cell = table.cell(i, j)
                cell.text = cell_text
                para = cell.text_frame.paragraphs[0]
                para.font.size = Pt(14)
                if i == 0:  # Header row
                    para.font.bold = True
                    cell.fill.solid()
                    cell.fill.fore_color.rgb = RGBColor(0, 51, 102)
                    para.font.color.rgb = RGBColor(255, 255, 255)
        
        # Add total
        total_box = slide.shapes.add_textbox(Inches(1), top + height + Inches(0.3), Inches(4), Inches(0.5))
        total_box.text_frame.paragraphs[0].text = f"Total Incidents: {total}"
        total_box.text_frame.paragraphs[0].font.size = Pt(16)
        total_box.text_frame.paragraphs[0].font.bold = True
    
    # Save
    prs.save(output_path)
    print(f"✅ PowerPoint saved: {os.path.abspath(output_path)}")
    print(f"📊 Contains {len(entities)} entity slides with tables")

# Export button and function
export_button = widgets.Button(description="Export to PPTX")
export_output = widgets.Output()

def on_export_clicked(b):
    with export_output:
        export_output.clear_output()
        if hasattr(entity_dropdown, 'df'):
            df_export = entity_dropdown.df
            entities_list = sorted(df_export['Entité'].dropna().unique())
            export_to_ppt(df_export, entities_list, "stable_analysis.pptx")
        else:
            print("❌ No data loaded. Please run previous cells first.")

export_button.on_click(on_export_clicked)

display(widgets.HTML("<b>Click to export all entity data to PowerPoint:</b>"), export_button, export_output)

HTML(value='<b>Click to export all entity data to PowerPoint:</b>')

Button(description='Export to PPTX', style=ButtonStyle())

Output()

# Export to PowerPoint (Single Table with Filter)
Creates one slide with all entity data in a single table. Uses VBA for dynamic filtering.

In [ ]:
def export_single_table_ppt(df, entities, output_path="stable_analysis_filter.pptx"):
    """Export all entity data to a single PowerPoint slide with VBA filter."""
    
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.dml.color import RGBColor
    import os
    
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    
    # Title slide
    title_slide_layout = prs.slide_layouts[0]
    slide = prs.slides.add_slide(title_slide_layout)
    slide.shapes.title.text = "Stable Analysis - All Entities"
    slide.placeholders[1].text = f"Total: {len(df)} incidents | {len(entities)} entities"
    
    # Data slide with table
    blank_layout = prs.slide_layouts[6]
    data_slide = prs.slides.add_slide(blank_layout)
    
    # Add title
    title_box = data_slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12), Inches(0.6))
    title_box.text_frame.paragraphs[0].text = "Incident Categories by Entity"
    title_box.text_frame.paragraphs[0].font.size = Pt(24)
    title_box.text_frame.paragraphs[0].font.bold = True
    title_box.text_frame.paragraphs[0].font.color.rgb = RGBColor(0, 51, 102)
    
    # Category order and display map
    category_order = ['CPU Issue', 'Memory Issue', 'Disk', 'Network', 'Nutanix', 'OS / Autres']
    display_map = {
        'CPU Issue': 'CPU Issue',
        'Memory Issue': 'Memory Issue',
        'OS / Autres': 'OS service',
        'Nutanix': 'Nutanix Issue',
        'Disk': 'VM availability / Disk / MSSQL / autres',
        'Network': 'Network',
    }
    
    # Build table data: Entity | CPU | Memory | Disk | Network | Nutanix | OS | Total
    table_rows = len(entities) + 1  # +1 for header
    table_cols = 8
    left = Inches(0.5)
    top = Inches(1.2)
    width = Inches(12.333)
    height = Inches(0.6 * table_rows)
    
    table = data_slide.shapes.add_table(table_rows, table_cols, left, top, width, height).table
    
    # Header row
    headers = ['Entity', 'CPU Issue', 'Memory Issue', 'VM avail/Disk/MSSQL', 'Network', 'Nutanix Issue', 'OS service', 'Total']
    for j, header in enumerate(headers):
        cell = table.cell(0, j)
        cell.text = header
        para = cell.text_frame.paragraphs[0]
        para.font.size = Pt(11)
        para.font.bold = True
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(0, 51, 102)
        para.font.color.rgb = RGBColor(255, 255, 255)
    
    # Data rows
    for i, entity in enumerate(entities):
        entity_df = df[df['Entité'] == entity]
        counts = entity_df['Categorie'].value_counts().reindex(category_order, fill_value=0)
        total = counts.sum()
        
        row_data = [
            entity,
            str(counts.get('CPU Issue', 0)),
            str(counts.get('Memory Issue', 0)),
            str(counts.get('Disk', 0)),
            str(counts.get('Network', 0)),
            str(counts.get('Nutanix', 0)),
            str(counts.get('OS / Autres', 0)),
            str(total)
        ]
        
        for j, cell_text in enumerate(row_data):
            cell = table.cell(i + 1, j)
            cell.text = cell_text
            cell.text_frame.paragraphs[0].font.size = Pt(10)
    
    # Add VBA for filtering
    vba_code = """Sub FilterByEntity()
    Dim shp As Shape
    Dim tbl As Table
    Dim i As Integer
    Dim selectedEntity As String
    
    'Get selected entity from input box
    selectedEntity = InputBox("Enter Entity name to filter:" & vbCrLf & "Leave empty to show all", "Filter Table", "")
    
    For Each shp In ActivePresentation.Slides(2).Shapes
        If shp.HasTable Then
            Set tbl = shp.Table
            For i = 2 To tbl.Rows.Count
                If selectedEntity = "" Or tbl.Cell(i, 1).Shape.TextFrame.TextRange.Text = selectedEntity Then
                    tbl.Rows(i).Visible = msoTrue
                Else
                    tbl.Rows(i).Visible = msoFalse
                End If
            Next i
        End If
    Next
End Sub
"""
    
    # Add instruction text
    instr_box = data_slide.shapes.add_textbox(Inches(0.5), top + height + Inches(0.3), Inches(12), Inches(0.5))
    instr_box.text_frame.paragraphs[0].text = "💡 To filter: Right-click table → Insert Table Style → Or use VBA macro (Alt+F8)"
    instr_box.text_frame.paragraphs[0].font.size = Pt(11)
    instr_box.text_frame.paragraphs[0].font.italic = True
    
    prs.save(output_path)
    print(f"✅ PowerPoint saved: {os.path.abspath(output_path)}")
    print(f"📊 Single table with {len(entities)} entities - use Excel for advanced filtering")

# Export button for single table
export_single_btn = widgets.Button(description="Export Single Table")
single_output = widgets.Output()

def on_single_export(b):
    with single_output:
        single_output.clear_output()
        if hasattr(entity_dropdown, 'df'):
            df_exp = entity_dropdown.df
            ents = sorted(df_exp['Entité'].dropna().unique())
            export_single_table_ppt(df_exp, ents, "stable_analysis_filter.pptx")
        else:
            print("❌ No data loaded. Run previous cells first.")

export_single_btn.on_click(on_single_export)

display(widgets.HTML("<b>Export as single table (all entities):</b>"), export_single_btn, single_output)

HTML(value='<b>Export as single table (all entities):</b>')

Button(description='Export Single Table', style=ButtonStyle())

Output()